In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils, context_utils
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE, get_generate_prompt, get_generate_prompt_internal_doc
from bait.core.bait_utils import INTERVENTION_OPTION, get_model_name_or_path
from bait.core.confirmation_bias_intervention import ConfirmationBiasIntervention

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/main_experiments'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def create_contexts_internal(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, intervention_option):
    contexts_internal = []
    if not common_utils.check_option(intervention_option, INTERVENTION_OPTION.OFF):
        prompts_internal = []
        for data in datas:
            prompts_internal.append(get_generate_prompt_internal_doc(data['question']))

        generated_contexts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_internal, max_seq_length, max_new_tokens*2
        )

        for generated_context in generated_contexts:
            generated_context = generated_context.replace('context_internal', '').strip()
            generated_context = generated_context.replace('context_in', '').strip()
            if generated_context.startswith(':'):
                generated_context = generated_context[1:].strip()

            contexts_internal.append(generated_context)

    return contexts_internal

In [ ]:
def mix_contexts(contexts_fact_dict: dict, contexts_counter_dict: dict, ext_n_fact: int, ext_n_counter: int):
    if ext_n_fact <= len(contexts_fact_dict) and ext_n_counter <= len(contexts_counter_dict):
        ext_contexts_fact = random.sample(list(contexts_fact_dict.values()), ext_n_fact)
        ext_contexts_counter = random.sample(list(contexts_counter_dict.values()), ext_n_counter)

        # 셔플 전 각각의 컨텍스트에 태그(출처)를 붙여 튜플 형태로 결합
        tagged_contexts = [(ctx, 'fact') for ctx in ext_contexts_fact] + [(ctx, 'counter') for ctx in ext_contexts_counter]

        # 태그를 붙인 상태에서 셔플
        random.shuffle(tagged_contexts)

        # 태그를 제거하고 위치 기록
        mixed_contexts = []
        fact_idxs, counter_idxs = [], []

        for i, (ctx, tag) in enumerate(tagged_contexts):
            mixed_contexts.append(ctx)
            if tag == 'fact':
                fact_idxs.append(i)
            else:
                counter_idxs.append(i)

        return mixed_contexts, fact_idxs, counter_idxs

    return None, None, None

In [ ]:
def add_prompts(question: str, answer: str, contexts_fact: dict, contexts_counter: dict, context_internal: str,
                prompts: list, answers: list, states: list, mixed_contexts_list: list):

    for file_format in FILE_FORMATS:
        contexts_fact_dict = contexts_fact[file_format]
        contexts_counter_dict = contexts_counter[file_format]

        for i in range(CONTEXT_SIZE):
            mixed_contexts, fact_idxs, counter_idxs = mix_contexts(contexts_fact_dict, contexts_counter_dict, i, CONTEXT_SIZE-1-i)

            if mixed_contexts is None:
                prompts.append(get_generate_prompt(question))
                answers.append(answer)
                states.append(False)
                mixed_contexts_list.append([])
            else:
                if context_internal is not None:
                    mixed_contexts.append(context_internal)

                prompts.append(get_generate_prompt(question, mixed_contexts))
                answers.append(answer)
                states.append(True)
                mixed_contexts_list.append(mixed_contexts)

In [ ]:
def extract_target_tok_idxs(tokenizer: PreTrainedTokenizerFast, prompts: list, mixed_contexts_list: list, intervention_option):
    target_tok_idxs_list = []
    if not common_utils.check_option(intervention_option, INTERVENTION_OPTION.NOT):
        chat_prompts, inputs = model_utils.make_inputs(tokenizer, device, prompts, max_seq_length,
                                                       return_offsets_mapping=True, return_all=True)

        offset_mappings = inputs.pop('offset_mapping').cpu() # torch.Size([50, 923, 2]) == (Batch, Seq_len, 2)

        for idx, (prompt, mixed_contexts) in enumerate(zip(chat_prompts, mixed_contexts_list)):
            context_tok_idxs = context_utils.extract_context_tok_idxs(prompt, mixed_contexts, offset_mappings[idx])
            target_tok_idxs_list.append(context_tok_idxs.get(f'{len(mixed_contexts)-1}', []))

        del inputs
        del offset_mappings
        common_utils.clear_gpu_memory()

    return target_tok_idxs_list

In [ ]:
def experiment_context_ratio(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, intervention_option, target_layers, batch_size=1):
    data_size = len(datas)
    cnts = {}

    for i, datas_batch in enumerate(container_utils.chunks(datas, batch_size)):
        prompts_batch, answers_batch, states_batch, mixed_contexts_list_batch = [], [], [], []

        # 1. (개입 시에만) 내재 지식 문서 생성
        contexts_internal = create_contexts_internal(model, tokenizer, datas_batch, intervention_option)

        # 2. 최종 프롬프트 생성
        for idx, data in enumerate(datas_batch):
            question = data['question']
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            contexts_fact = data['contexts_fact']
            contexts_counter = data['contexts_counter']
            context_internal = contexts_internal[idx] if contexts_internal else None
            
            add_prompts(
                question, answer_fact, contexts_fact, contexts_counter, context_internal,
                prompts_batch, answers_batch, states_batch, mixed_contexts_list_batch
            )

        # 3. 개입 대상 토큰 인덱스 추출
        target_tok_idxs_list_batch = extract_target_tok_idxs(tokenizer, prompts_batch, mixed_contexts_list_batch, intervention_option)

        # 4. 실제 forward 과정에서 개입을 위한 Hook 자동 등록/해제
        with ConfirmationBiasIntervention(model, intervention_option, target_layers) as cbi:
            cbi.set_target_toks(target_tok_idxs_list_batch)

            generated_texts = model_utils.get_generated_texts(
                model, tokenizer, device,
                prompts_batch, max_seq_length, max_new_tokens
            )

        # 5. 결과 확인 성능 측정
        idx = -1
        for j in range(batch_size):
            for file_format in FILE_FORMATS:
                for k in range(CONTEXT_SIZE):
                    idx += 1

                    ext_n_fact = idx % CONTEXT_SIZE
                    ext_n_counter = CONTEXT_SIZE - 1 - ext_n_fact

                    # key 저장 용도
                    container_utils.add_str_int(cnts, f'{file_format}\t{ext_n_fact}\t{ext_n_counter}', 0)
                    container_utils.add_str_int(cnts, f'ALL\t{ext_n_fact}\t{ext_n_counter}', 0)

                    if not states_batch[idx]:
                        container_utils.add_str_int(cnts, f'{file_format}_skip\t{ext_n_fact}\t{ext_n_counter}', 1)
                        container_utils.add_str_int(cnts, f'ALL_skip\t{ext_n_fact}\t{ext_n_counter}', 1)
                    else:
                        if model_utils.is_correct(generated_texts[idx], answers_batch[idx])[1]:
                            container_utils.add_str_int(cnts, f'{file_format}\t{ext_n_fact}\t{ext_n_counter}', 1)
                            container_utils.add_str_int(cnts, f'ALL\t{ext_n_fact}\t{ext_n_counter}', 1)

        if (i+1) % 100 == 0:
            print(f'experiment_context_ratio() {(i+1)*batch_size} complet.')
    print(f'experiment_context_ratio() {data_size} complet.\n')





    file_formats = FILE_FORMATS + ['ALL']
    for file_format in file_formats:
        for i in range(CONTEXT_SIZE):
            cnt_key = f'{file_format}\t{CONTEXT_SIZE-1-i}\t{i}'
            cnt_skip_key = f'{file_format}_skip\t{CONTEXT_SIZE-1-i}\t{i}'

            if cnt_key in cnts.keys():
                cnt_value = cnts[cnt_key]

                if cnt_key.startswith('ALL'):
                    size = data_size * len(FILE_FORMATS)
                else:
                    size = data_size

                if cnt_skip_key in cnts.keys():
                    size -= cnts[cnt_skip_key]

                print(f'{cnt_key}\t{cnt_value}\t{cnt_value}/{size}\t{cnt_value/size}')
        print()

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']
intervention_option = INTERVENTION_OPTION.ALL

for model_name in model_names:
    model_name_or_path = get_model_name_or_path(model_name)
    
    model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

    # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

    # model = None
    # tokenizer = None

    for zero_shot in ['fact', 'counter', 'other']:
        in_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
        datas = json_utils.load_json(in_file_path)

        experiment_context_ratio(model, tokenizer, datas, intervention_option, [-2])

    del model
    del tokenizer
    common_utils.clear_gpu_memory()